In [ ]:
from llama_index.readers.file.unstructured import UnstructuredReader
from unstructured.partition.auto import partition
from llama_index.core import Document
from pathlib import Path


def smart_load(file_path):
    """
    智能文档加载器：根据文件类型选择最佳解析策略

    Args:
        file_path: 文件路径

    Returns:
        解析后的Document对象列表  
    """
    file_path = Path(file_path)
    file_ext = file_path.suffix.lower()

    # 定义复杂文件类型（需要高精度解析）
    complex_types = {
        ".pdf",  # PDF文档（可能包含表格、图像、复杂布局）
        ".png",
        ".jpg",
        ".jpeg",
        ".gif",
        ".bmp",
        ".tiff",  # 图片文件（需要OCR）
        ".docx",
        ".doc",  # Word文档（可能包含复杂格式）
        ".pptx",
        ".ppt",  # PowerPoint（复杂布局）
        ".xlsx",
        ".xls",  # Excel（表格结构）
    }

    # 简单文件类型（可以用Reader直接处理）
    simple_types = {".txt", ".md", ".csv", ".html", ".xml", ".json"}

    if file_ext in complex_types:
        # 复杂文件使用底层解析，获得更好的结构识别
        print(f"检测到复杂文件类型 {file_ext}，使用partition高精度解析")
        try:
            elements = partition(
                filename=str(file_path),
                # 使用hi_res模式进行高精度解析
                strategy="hi_res",
                # 支持中文、英文
                languages=["eng", "chi_sim"],
                # 推断表格结构
                infer_table_structure=True,
            )
            # 将解析元素转换为Document对象
            return [
                Document(
                    text=e.text,
                    metadata={
                        "source": str(file_path),
                        "element_type": type(e).__name__,
                        "file_type": file_ext,
                    },
                )
                for e in elements
                if e.text.strip()
            ]  # 过滤空文本
        except Exception as e:
            print(f"高精度解析失败，回退到Reader: {e}")
            # 回退到Reader
            reader = UnstructuredReader()
            return reader.load_data(file=file_path)

    else:
        # 简单文件或未知类型优先使用Reader
        print(f"检测到简单文件类型 {file_ext}，使用Reader解析")
        try:
            # 直接使用Reader进行简单解析
            reader = UnstructuredReader()
            # 加载解析后的文档，返回 Document 对象列表
            docs = reader.load_data(file=file_path)
            return docs
        except Exception as e:
            print(f"Reader解析失败，回退到partition: {e}")
            # 回退到底层解析
            elements = partition(filename=str(file_path), strategy="auto")
            # return [Document(text=e.text, metadata={"source": str(file_path)}) for e in elements]
            # 推荐的安全改法
            docs = []
            for e in elements:
                if not e.text.strip():
                    continue
                meta = e.metadata.to_dict() if hasattr(e, "metadata") else {}
                meta.update(
                    {
                        "source": str(file_path),
                        "element_type": type(e).__name__,
                        "file_type": file_ext,
                    }
                )
                docs.append(Document(text=e.text, metadata=meta))
            return docs

In [2]:
documents = smart_load("甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf")

检测到复杂文件类型 .pdf，使用partition高精度解析


2026-06-25 10:57:10.343876596 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+
2026-06-25 10:58:25,076 - INFO - pikepdf C++ to Python logger bridge initialized
2026-06-25 10:58:28,396 - INFO - HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-06-25 10:58:28,651 - INFO - HTTP Request: HEAD https://huggingface.co/unstructuredio/yolo_x_layout/resolve/main/yolox_l0.05.onnx "HTTP/1.1 302 Found"
2026-06-25 10:58:28,652 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-25 10:58:49,303 - INFO - Reading PDF for file: 甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf ...
2026-06-25 1

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

2026-06-25 10:59:59,800 - INFO - Table model successfully loaded to cpu


In [5]:
documents[8].text

'海外 AI 视角：（1）英伟达推出 B200A，2025 年 Blackwell GPU 有望 上量。《科创板日报》8 月 7 日讯，据 TrendForce 集邦咨询，英伟达仍 计划在 2024 年下半年推出 B100 及 B200，供应 CSPs（云端服务业者） 客户，并规划于 2024 年第三季后陆续供货。在 CoWoS-L 良率和量产 尚待整备的情况下，英伟达同步规划降规版 B200A 给其他企业客户， 并转为采用 CoWoS-S 封装技术。B200A 的存储器规格将采用 4 颗 HBM3e（第五代高带宽内存）12hi（12 层堆叠），总容量为 144GB。 预期 OEMs（原始设备制造商）应会于 2025 年上半年正式拿到 B200A 芯片。到 2025 年 Blackwell 平台将占英伟达高端 GPU 逾八成，并促 使英伟达高端 GPU 系列的出货年增率上升至 55%。（2）马斯克旗下 xAI 公司发布 Grok-2 测试版。《科创板日报》8 月 14 日讯，马斯克旗 下 xAI 正式发布语言模型 Grok 2 早期预览版。据介绍，该系列模型具 有聊天、编码和推理等功能，包括 Grok 2 和 Grok 2 mini 两个版本， 目前正在 X 平台上进行测试，并将于本月晚些时候通过企业 API 提供 上述模型。（3）苹果计划推出 AI 桌面机器人。据环球网援引彭博社报 道，苹果公司正在加速其桌面机器人项目的研发工作，并计划最早于 2026 年推出这款创新设备。据报道，苹果的桌面机器人将配备一块类 似 iPad 的大尺寸显示屏，由一个纤薄的机械臂支撑，能够实现上下倾 斜和 360 度旋转。这款设备将集成智能家居控制中心、视频会议终端 和家庭安全监控等多项功能。桌面机器人将搭载 Siri 和 Apple Intelligence 技术，具备响应多种语音指令、识别不同声音的能力，并 能自动调整显示屏方向以面向房间内的用户，提供更加智能化和个性 化的交互体验。'

In [1]:
from unstructured.partition.pdf import partition_pdf

# 使用 partition_pdf 函数解析 PDF 文档 为elements类型
elements = partition_pdf(
    filename="甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf",
    strategy="hi_res",  # 使用高精度模式
    extract_images_in_pdf=False,
)

2026-06-25 11:24:06.100606182 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+
No languages specified, defaulting to English.


In [5]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Document
from typing import List
import json

# ============= 步骤1: 预处理unstructured elements =============
def create_documents_with_title_context(elements) -> List[Document]:
    """
    将unstructured elements转换为Documents,并将Title信息注入metadata
    """
    documents = []
    current_title_hierarchy = {
        "h1": "",
        "h2": "",
        "h3": "",
    }

    accumulated_text = []
    accumulated_metadata = {}

    for elem in elements:
        elem_type = type(elem).__name__
        elem_metadata = elem.metadata.to_dict()

        # 如果是Title,更新层级信息
        if elem_type == "Title":
            # 如果有累积的文本,先创建Document
            if accumulated_text:
                doc = Document(
                    text="\n\n".join(accumulated_text),
                    metadata=accumulated_metadata.copy()
                )
                documents.append(doc)
                accumulated_text = []

            # 简单的层级判断(可根据实际情况改进)
            title_text = elem.text
            if len(title_text) < 20:  # 短标题可能是低层级
                current_title_hierarchy["h3"] = title_text
            elif len(title_text) < 40:
                current_title_hierarchy["h2"] = title_text
                current_title_hierarchy["h3"] = ""
            else:
                current_title_hierarchy["h1"] = title_text
                current_title_hierarchy["h2"] = ""
                current_title_hierarchy["h3"] = ""

            # 初始化新section的metadata
            accumulated_metadata = {
                **elem_metadata,
                "section_title": title_text,
                "title_h1": current_title_hierarchy["h1"],
                "title_h2": current_title_hierarchy["h2"],
                "title_h3": current_title_hierarchy["h3"],
            }

            # Title本身也加入文本
            accumulated_text.append(f"# {title_text}")

        else:
            # 非Title元素,累积到当前section
            if not accumulated_metadata:
                # 如果还没有metadata(文档开头没有Title的情况)
                accumulated_metadata = {
                    **elem_metadata,
                    "section_title": "前言",
                }

            accumulated_text.append(elem.text)

            # 更新metadata(保留最新的page_number等信息)
            accumulated_metadata.update({
                k: v for k, v in elem_metadata.items()
                if k in ['page_number', 'filename']
            })

    # 处理最后累积的文本
    if accumulated_text:
        doc = Document(
            text="\n\n".join(accumulated_text),
            metadata=accumulated_metadata
        )
        documents.append(doc)

    return documents

# ============= 步骤2: 应用自定义处理 =============
enriched_documents = create_documents_with_title_context(elements)

print("=" * 80)
print("Metadata增强的Documents")
print("=" * 80)
for i, doc in enumerate(enriched_documents[:3]):
    print(f"\n--- Document {i+1} ---")
    print(f"文本长度: {len(doc.text)} 字符")
    print(f"文本预览:\n{doc.text[:150]}...")
    print(f"\nMetadata:")
    for key, value in doc.metadata.items():
        print(f"  {key}: {value}")

# ============= 步骤3: 使用SentenceSplitter进一步切分 =============
"""
如果Documents还是太大,可以进一步切分
关键: metadata会自动继承到所有child nodes
"""

node_parser = SentenceSplitter(
    chunk_size=512,
    chunk_overlap=50,
    separator=" ",
)

nodes = node_parser.get_nodes_from_documents(enriched_documents)

print("\n" + "=" * 80)
print("进一步切分后的Nodes (metadata已继承)")
print("=" * 80)
print(f"总节点数: {len(nodes)}")
for i, node in enumerate(nodes[:29]):
    print(f"\n--- Node {i+1} ---")
    print(f"Node ID: {node.node_id}")
    print(f"文本预览: {node.text[:80]}...")
    print(f"继承的Metadata: {json.dumps(node.metadata, ensure_ascii=False, indent=2)}")


Metadata增强的Documents

--- Document 1 ---
文本长度: 36 字符
文本预览:
证 券 研 究 报 告 行 业 研 究

中小市值

行业研究/行业点评...

Metadata:
  detection_class_prob: 0.561924397945404
  is_extracted: partial
  coordinates: {'points': ((np.float64(24.472658157348633), np.float64(-4.65582799911499)), (np.float64(24.472658157348633), np.float64(1497.43310546875)), (np.float64(110.49360656738281), np.float64(1497.43310546875)), (np.float64(110.49360656738281), np.float64(-4.65582799911499))), 'system': 'PixelSpace', 'layout_width': 2894, 'layout_height': 4093}
  last_modified: 2026-06-22T11:08:02
  filetype: application/pdf
  languages: ['kor']
  page_number: 1
  filename: 甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf
  section_title: 前言

--- Document 2 ---
文本长度: 1076 字符
文本预览:
# 海外科技巨头持续发力 AI，龙头公司中报业绩亮眼

——AI 行业点评报告

行业： 日期： 中小市值 yxzqdatemark 2024年08月20日 分析师： 彭毅 E-mail： pengyi@yongxingsec.c om SAC编号： S1760523090003 分析师： 张恬 E...

Metadata:
  detection_class_prob: 0.4599014222621918
  is_extracted: true
  coordinates: {'points